# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is a Dataset object, not a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and associated fields, referencing them by their @id
def overview_of_recordsets(dataset):
    print("Available Record Sets:")
    record_sets = list(dataset.record_sets)
    for rs in record_sets:
        print(f'- RecordSet @id: {rs["@id"]}')
        print(f'  Name: {rs["name"]}')
        print(f'  Description: {rs.get("description", "No description")}' )
        # Print fields
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        elif fields is None:
            fields = []
        print(f'  Fields:')
        for field in fields:
            # Try to fetch the full field object (sometimes field is just a string @id)
            if isinstance(field, dict):
                print(f'    - Field @id: {field.get("@id", field)} | name: {field.get("name") or "?"}')
            elif isinstance(field, str):
                print(f'    - Field @id: {field}')
        print()

overview_of_recordsets(dataset)

# For reference within the notebook, select the record set(s) by their @id below after running this cell.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# ---
# Replace these IDs with the ones you want to analyze, from the previous output:
record_sets = [
    # Example: 'https://api.app.sen.science/frontiers/7154140/b91ced1e-7fc1-41ef-8e4b-ba31ac16d2b0'
]
if not record_sets:
    print("⚠️ Please populate `record_sets` with actual record set @id(s) from Section 2 above to proceed.")
else:
    dataframes = {}
    for record_set in record_sets:
        records = list(dataset.records(record_set=record_set))
        if not records:
            print(f'Record set {record_set}: No records found or there was a loading error.')
        else:
            dataframes[record_set] = pd.DataFrame(records)
            print(f'Loaded DataFrame for {record_set} with columns:')
            print(dataframes[record_set].columns.tolist())
            display(dataframes[record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Select the record set and field IDs for EDA:
# Example:
#   record_set_id = 'https://api.app.sen.science/frontiers/7154140/b91ced1e-7fc1-41ef-8e4b-ba31ac16d2b0'
#   numeric_field_id = 'abundance_mean'  # Use actual @id or column name from DataFrame
#   group_field_id = 'sample_type'       # Use actual @id or column name from DataFrame

record_set_id = None
numeric_field_id = None
group_field_id = None

# ---
if not all([record_set_id, numeric_field_id]):
    print("⚠️ Please define `record_set_id` and `numeric_field_id` above based on previous DataFrame column output.")
elif record_set_id not in dataframes:
    print(f"⚠️ DataFrame for record set {record_set_id} not loaded.")
elif numeric_field_id not in dataframes[record_set_id].columns:
    print(f"⚠️ Numeric field '{numeric_field_id}' not found in columns of record set {record_set_id}.")
else:
    df = dataframes[record_set_id]
    # Ensure numeric data
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10  # Adjust threshold as needed for your field
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    mean_ = filtered_df[numeric_field_id].mean()
    std_ = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_) / std_
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Optionally group by a categorical field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: histogram and boxplot of a selected numeric column (ensure prior cells are configured)
import matplotlib.pyplot as plt
import seaborn as sns

if not all([record_set_id, numeric_field_id]):
    print("⚠️ Please define `record_set_id` and `numeric_field_id` above to visualize data.")
elif record_set_id not in dataframes:
    print(f"⚠️ DataFrame for record set {record_set_id} not loaded.")
elif numeric_field_id not in dataframes[record_set_id].columns:
    print(f"⚠️ Numeric field '{numeric_field_id}' not found in columns of record set {record_set_id}.")
else:
    plt.figure(figsize=(12,5))
    df = dataframes[record_set_id]
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*Write your summary of findings here. Discuss the data, trends, quality, or anything notable found in your analysis, referencing field @id's where relevant.*